# Introduction

In [ ]:
import random

class RandomWalker:
    def __init__(self):
        self.position = 0

    def walk(self, n):
        self.position = 0
        for i in range(n):
            yield self.position
            self.position += 2*random.randint(0,1) - 1
    
walker = RandomWalker()
%timeit [position for position in walker.walk(10000)]

In [ ]:
def random_walk(n):
    position = 0
    walk = [position]
    for i in range(n):
        position += 2 * random.randint(0,1)-1
        walk.append(position)
    return walk

%timeit random_walk(10000)

1.63 ms ± 17.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
def random_walk_faster(n=10000):
    from itertools import accumulate
    steps = random.choices([-1, +1], k=n)
    return [0] + list(accumulate(steps))

%timeit random_walk_faster(10000)

266 μs ± 1.05 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
import numpy as np

def random_walk_numpy(n):
    steps = np.random.choice([-1,+1], n)
    return np.cumsum(steps)

%timeit random_walk_numpy(10000)

30.7 μs ± 144 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


# Arrays

In [ ]:
Z = np.ones(4 * 1000000, np.float32)
Z[...]=0
Z

array([0., 0., 0., ..., 0., 0., 0.], shape=(4000000,), dtype=float32)

In [ ]:
%timeit Z.view(np.float16)[...] = 0
%timeit Z.view(np.int16)[...] = 0
%timeit Z.view(np.int32)[...] = 0
%timeit Z.view(np.float32)[...] = 0
%timeit Z.view(np.float64)[...] = 0
%timeit Z.view(np.complex128)[...] = 0
%timeit Z.view(np.int8)[...] = 0

113 μs ± 370 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
113 μs ± 336 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
113 μs ± 436 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
113 μs ± 468 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
114 μs ± 442 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
229 μs ± 913 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
57.1 μs ± 172 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


An instance of class `ndarray` consists of a contiguous one-dimensional segment of computer memory (owned by the array, or by some other object), combined with an indexing scheme that maps `N` integers into the location of an item in the block.

In [ ]:
Z = np.arange(9).reshape(3, 3).astype(np.int16)
print(Z.itemsize)
print(Z.shape)
print(Z.ndim)

2
(3, 3)
2


In [ ]:
strides = Z.shape[1] * Z.itemsize, Z.itemsize
print(strides)
print(Z.strides)

(6, 2)
(6, 2)


In [ ]:
index = 1,1
print(Z[index].tobytes())

offset_start = 0
for i in range(Z.ndim):
    offset_start += Z.strides[i] * index[i]
offset_end = offset_start + Z.itemsize
print(Z.tobytes()[offset_start:offset_end])

b'\x04\x00'
b'\x04\x00'


In [ ]:
V = Z[::2, ::2]
V

array([[0, 2],
       [6, 8]], dtype=int16)

In [ ]:
Z = np.zeros(9)
Z_view = Z[:3]
Z_view[...] = 1
Z

array([1., 1., 1., 0., 0., 0., 0., 0., 0.])

In [ ]:
Z = np.zeros(9)
Z_copy = Z[[0,1,2]]
Z_copy[...]=1
Z

array([0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
Z = np.random.uniform(0, 1, (5, 5))
Z1 = Z[:3,:]
Z2 = Z[[0, 1, 2], :]
print(np.allclose(Z1,Z2))
print(Z1.base is Z)
print(Z2.base is Z)
print(Z2.base is None)

True
True
False
True


In [ ]:
Z = np.zeros((5,5))
print(Z.ravel().base is Z )
print(Z[::2, ::2].ravel().base is Z) # not possible to return a view since it needs to be a flat, C-contiguous 1D array
print(Z.flatten().base is Z)

True
False
False


In [ ]:
X = np.ones(10, dtype=np.int16)
Y = np.ones(10, dtype=np.int16)
A = 2*X + 2*Y

In [ ]:
Z1 = np.arange(10)
Z2 = Z1[1:-1:2]

In [ ]:
step = Z2.strides[0] // Z1.strides[0]
print(step)

2


In [ ]:
offset_start = np.lib.array_utils.byte_bounds(Z2)[0] - np.lib.array_utils.byte_bounds(Z1)[0]
offset_start

8

In [ ]:
offset_stop = np.lib.array_utils.byte_bounds(Z2)[-1] - np.lib.array_utils.byte_bounds(Z1)[-1]
offset_stop

-16

In [ ]:
start = offset_start // Z1.itemsize
stop = Z1.size + offset_stop // Z1.itemsize
print(start, stop, step)

1 8 2


In [ ]:
print(np.allclose(Z1[start:stop:step], Z2))

True


In [ ]:
# Negative Steps
byte_bounds = np.lib.array_utils.byte_bounds

Z1 = np.arange(10)
Z2 = Z1[-1:1:-2]
print(f"Z1={Z1}")
print(f"Z2={Z2}")

step = Z2.strides[0] // Z1.strides[0]
print(step)

if step > 0:
    offset_start = byte_bounds(Z2)[0] - byte_bounds(Z1)[0]
    offset_end = byte_bounds(Z2)[1] - byte_bounds(Z1)[0] - Z2.itemsize
else:
    offset_start = byte_bounds(Z2)[1] - byte_bounds(Z1)[0] - Z2.itemsize
    offset_end = byte_bounds(Z2)[0] - byte_bounds(Z1)[0]

print(offset_start, offset_end)

start = offset_start // Z2.itemsize
stop = offset_end // Z2.itemsize + (1 if step > 0 else -1)
print(start, stop, step)

Z1[start:stop:step]

Z1=[0 1 2 3 4 5 6 7 8 9]
Z2=[9 7 5 3]
-2
72 24
9 2 -2


array([9, 7, 5, 3])

In [ ]:
# Multidimensional
byte_bounds = np.lib.array_utils.byte_bounds

Z1 = np.arange(12).reshape(3, 4)
Z2 = Z1[1:-1:2]
print(Z1)
print(Z2)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[4 5 6 7]]
